# R4 · 3 · MTGNN condicionado ao tamanho da temporada (GPU)

**Antes de rodar (uma vez só)**, em `MyDrive/dengue_V4/` (que já tem tudo das rodadas 1 a 3):
1. suba **`r4_lib.py`** (pasta `data_imdc_2026/RODADA4/`) e confira que `r3_lib.py` e `r2_lib.py` estão lá;
2. as pastas `resultados_uf_r4`, `revisoes_r4`, `cache_r4` e `avaliacao_r4` são criadas sozinhas.

**Usa a rodada 3:** o normal (ENS_grafos3_λ) vem de `cache_r3/normal_ens_grafos3_bb_clim.parquet` e as misturas da R3 entram na comparação.

**Complementar:** nada das rodadas anteriores muda. As previsões oficiais (corte de junho) vão para `resultados_uf_r4/uf/`; as revisões de julho a outubro, para `revisoes_r4/uf/`; as tabelas, para `avaliacao_r4/` (com data e hora no nome e uma cópia `_ultimo`).

**Retomada:** se a internet ou a luz cair, reconecte e **rode tudo de novo**. Cada etapa grava o seu arquivo (mar, sorotipo por ano, variáveis, detector por método × corte, parâmetros, um arquivo por modelo × temporada, os pesos das redes a cada 5 épocas, um PNG por UF) e o que já existe é pulado.

GPU **L4** (ou A100), 2–4 h para 8 temporadas × 5 sementes. **Rode depois do R4_1.** Pode rodar **ao mesmo tempo** que o R4_2. Os pesos das redes são gravados a cada 5 épocas em `cache_r4/ckpt/` e o treino retoma de onde parou.

**O que muda em relação à MTGNN_pesopop da rodada 2** (mesma rede, mesmas entradas, mesma perda ponderada pela população):
1. **Condição:** duas entradas novas, o tamanho relativo da temporada principal da janela e um indicador de "conhecido". No treino, entra o tamanho real das temporadas já terminadas.
2. **Cópias sem condição (20%):** a mesma rede também aprende a prever sem saber o tamanho. Essa versão é a `MTGNN_aug_UF`.
3. **Aumento com anos grandes sintéticos (60%):**
   - a temporada principal do futuro é multiplicada por *k* entre 2 e 8, com o passado igual e a condição + log *k*;
   - assim a rede aprende a desenhar temporadas maiores que as já vistas, inclusive a forma do pico (Al Bannoud et al. 2026: até −80% de erro em São Paulo 2024);
   - o teto de segurança sobe de 3× para 10× o maior valor já visto.
4. **Previsão:**
   - 24 cenários de tamanho tirados do detector ampliado (correlação de 0,6 entre UFs, porque anos grandes costumam ser regionais);
   - a rede roda uma vez por cenário e as trajetórias são misturadas;
   - resultado: `MTGNN_cond_UF`.

**Sem vazamento:** em cada temporada, a rede é treinada só com dados até a SE25. A condição de treino usa apenas temporadas terminadas até lá. Na previsão, o tamanho vem do detector, que também só viu o passado.


In [ ]:
NOTEBOOK = 'r4_mtgnn'

# @title 1 · Drive, arquivos e pacotes
import os, sys, shutil, time, glob, subprocess, json, traceback
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_DRIVE, PASTA_LOCAL, NO_COLAB = '/content/drive/MyDrive/dengue_V4', '/content/r4', True
except ImportError:
    PASTA_DRIVE = PASTA_LOCAL = os.path.abspath('.'); NO_COLAB = False
if NO_COLAB:
    os.makedirs(PASTA_LOCAL, exist_ok=True)
    for item in ('v4_lib.py', 'comp_lib.py', 'comp_modelos.py', 'uf_lib.py', 'semanal_lib.py', 'r2_lib.py', 'r3_lib.py',
                 'r4_lib.py', 'dados', 'dados_uf', 'referencias_imdc3'):
        o, d = os.path.join(PASTA_DRIVE, item), os.path.join(PASTA_LOCAL, item)
        assert os.path.exists(o), f'Faltou subir {item} para {PASTA_DRIVE}'
        shutil.copytree(o, d, dirs_exist_ok=True) if os.path.isdir(o) else shutil.copy(o, d)
    pass
sys.path.insert(0, PASTA_LOCAL)
PASTA_R4 = os.path.join(PASTA_DRIVE, 'resultados_uf_r4')
PASTA_REV4 = os.path.join(PASTA_DRIVE, 'revisoes_r4')
PASTA_C4 = os.path.join(PASTA_DRIVE, 'cache_r4')
PASTA_AV4 = os.path.join(PASTA_DRIVE, 'avaliacao_r4')
PASTA_R3 = os.path.join(PASTA_DRIVE, 'resultados_uf_r3')
PASTA_C3 = os.path.join(PASTA_DRIVE, 'cache_r3')
PASTA_AV2 = os.path.join(PASTA_DRIVE, 'avaliacao_r2')
PASTA_COMP = os.path.join(PASTA_DRIVE, 'resultados_competicao')
PASTA_EXT = os.path.join(PASTA_DRIVE, 'resultados_externos')
for p in (os.path.join(PASTA_R4, 'uf'), os.path.join(PASTA_REV4, 'uf'), PASTA_C4, os.path.join(PASTA_C4, 'ckpt'), PASTA_AV4):
    os.makedirs(p, exist_ok=True)
import importlib, v4_lib as V, comp_lib as L, comp_modelos as CM, uf_lib as U, semanal_lib as S, r2_lib as R2, r3_lib as R3, r4_lib as R4
for m in (V, L, CM, U, S, R2, R3, R4):
    importlib.reload(m)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)
ctx = U.ContextoUF(os.path.join(PASTA_LOCAL, 'dados'), os.path.join(PASTA_LOCAL, 'referencias_imdc3'),
                   os.path.join(PASTA_LOCAL, 'dados_uf'))
D = ctx.Du
TUNING, TESTE = U.TUNING, U.TESTE
REFAZER = False
R4.PESO_ABS = 0.5     # critério misto = (1 − p)·R × referência + p·WIS absoluto relativo. Suba (ex.: 0.7) para priorizar a altura dos picos


In [ ]:
# @title 2 · Detector ampliado e configuração
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '⚠️ sem GPU (vai ser MUITO lento)')
ESC, DET = R4.detector_escolhido4(PASTA_C4)
SEMENTES = (1, 2, 3, 4, 5)
REDE = U.MTGNN(sementes=SEMENTES)
CK = os.path.join(PASTA_C4, 'ckpt')
NOMES = ('MTGNN_cond_UF', 'MTGNN_aug_UF')


In [ ]:
# @title 3 · Treinar e prever (uma temporada por vez; pula o que já existe)
for s in TUNING + TESTE:
    if not REFAZER and all(os.path.exists(L._arq(PASTA_R4, 'uf', n, s)) for n in NOMES):
        continue
    t0 = time.time()
    c, datas, idx, h = CM._horizontes(ctx, s)
    redes, dd = R4.treinar_cond(ctx, REDE, c, 68, CK, f's{s}')
    datas, saida = R4.prever_temporada_cond(ctx, redes, dd, REDE, s, DET)
    for nome, q in saida.items():
        R4.grava_modelo(V.para_df(V.arruma_quantis(q), datas, D.unidades, nome, s), PASTA_R4, nome)
    del redes, dd
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    print(f'✅ {s}: {list(saida)} · {(time.time() - t0) / 60:.1f} min', flush=True)


In [ ]:
# @title 4 · Conferência rápida no teste (o veredito completo é no R4_4)
NORMAL = R4.ler_normal(ctx, PASTA_DRIVE, PASTA_AV2, PASTA_COMP, PASTA_EXT, PASTA_C3, TUNING + TESTE)
X = pd.concat([NORMAL, R4.ler_pastas(PASTA_R4)], ignore_index=True)
X = X[X.season.isin(TESTE) & X.modelo.isin(['ENS_grafos3_λ', 'BB'] + list(NOMES))]
R4.criterio(ctx, X, ref='BB').round(3)
